# Daily Challenge – Week 6, Day 3: Stock Price Prediction with LSTM (PyTorch)
Predict the **next day's closing price** of a stock from the previous days. Run in **Google Colab** (PyTorch is pre-installed; a GPU is optional).

## 1. Install / import the required libraries
(`gensim` and `spacy` are listed in the instructions but aren't needed for this task, so we skip them.)

In [ ]:
# !pip install kagglehub      # only needed outside Colab
import os, glob, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score, mean_absolute_error

torch.manual_seed(42)
np.random.seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)

## 2. Load and Preprocess the Dataset
The Kaggle *Stock Market Dataset* has one CSV per company. We use **Apple (AAPL)**. `kagglehub` downloads public Kaggle datasets directly in Colab.

In [ ]:
TICKER = "AAPL"

def load_stock(ticker):
    try:
        import kagglehub
        path = kagglehub.dataset_download("jacksoncrow/stock-market-dataset")
        files = glob.glob(os.path.join(path, "**", f"{ticker}.csv"), recursive=True)
        return pd.read_csv(files[0])
    except Exception as e:
        print("Kaggle download failed (", e, ") – falling back to Yahoo Finance via yfinance")
        import yfinance as yf
        data = yf.download(ticker, start="2000-01-01", end="2020-04-01", auto_adjust=False)
        data.columns = [c[0] if isinstance(c, tuple) else c for c in data.columns]
        return data.reset_index()

raw = load_stock(TICKER)
print(raw.shape)
raw.head()

In [ ]:
df = raw.copy()
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date").set_index("Date")

# Drop unnecessary columns: "Adj Close" duplicates Close for our purpose
df = df[["Open", "High", "Low", "Close", "Volume"]].dropna()

# Use the last 10 years, where prices are most relevant
df = df[df.index >= df.index.max() - pd.DateOffset(years=10)]

# Target = NEXT day's closing price
df["Target"] = df["Close"].shift(-1)
df = df.dropna()
print(df.shape)
df.tail()

In [ ]:
df["Close"].plot(figsize=(12, 4), title=f"{TICKER} closing price")
plt.ylabel("USD")
plt.show()

## 3. Prepare the Dataset for Training
Split **by time** (70% train, 15% validation, 15% test, no shuffling), then normalize with `MinMaxScaler` fitted on the **training period only**.

In [ ]:
FEATURES = ["Open", "High", "Low", "Close", "Volume"]
n = len(df)
train_end, val_end = int(n * 0.70), int(n * 0.85)

feature_scaler = MinMaxScaler().fit(df[FEATURES].iloc[:train_end])
target_scaler = MinMaxScaler().fit(df[["Target"]].iloc[:train_end])

X_scaled = feature_scaler.transform(df[FEATURES])
y_scaled = target_scaler.transform(df[["Target"]]).ravel()

class StockDataset(Dataset):
    """Each item: the previous `window` days of features → the next-day target."""
    def __init__(self, X, y, window=30):
        self.X, self.y, self.window = X, y, window
    def __len__(self):
        return len(self.X) - self.window + 1
    def __getitem__(self, i):
        x = self.X[i:i + self.window]
        target = self.y[i + self.window - 1]         # target of the last day in the window = next day's close
        return torch.tensor(x, dtype=torch.float32), torch.tensor(target, dtype=torch.float32)

WINDOW = 30
train_ds = StockDataset(X_scaled[:train_end], y_scaled[:train_end], WINDOW)
val_ds = StockDataset(X_scaled[train_end - WINDOW + 1:val_end], y_scaled[train_end - WINDOW + 1:val_end], WINDOW)
test_ds = StockDataset(X_scaled[val_end - WINDOW + 1:], y_scaled[val_end - WINDOW + 1:], WINDOW)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=64)
test_loader = DataLoader(test_ds, batch_size=64)
print("Samples – train:", len(train_ds), "| val:", len(val_ds), "| test:", len(test_ds))

## 4. Define the LSTM Model
Two stacked recurrent layers with dropout, followed by a dense layer. The `cell` argument lets us switch between **LSTM** and **GRU** (the instructions mention both).

In [ ]:
class StockRNN(nn.Module):
    def __init__(self, n_features, hidden_size=64, num_layers=2, dropout=0.2, cell="LSTM"):
        super().__init__()
        rnn_class = nn.LSTM if cell == "LSTM" else nn.GRU
        self.rnn = rnn_class(n_features, hidden_size, num_layers=num_layers,
                             batch_first=True, dropout=dropout)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.rnn(x)            # out: (batch, window, hidden)
        last = out[:, -1, :]            # hidden state of the last day
        return self.fc(self.dropout(last)).squeeze(-1)

model = StockRNN(n_features=len(FEATURES), cell="LSTM").to(device)
print(model)

## 5. Train the Model

In [ ]:
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
EPOCHS = 30

train_losses, val_losses = [], []
best_val, best_state = float("inf"), None

for epoch in range(EPOCHS):
    model.train()
    total = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
        total += loss.item() * len(xb)
    train_losses.append(total / len(train_ds))

    model.eval()
    total = 0.0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            total += criterion(model(xb), yb).item() * len(xb)
    val_losses.append(total / len(val_ds))

    if val_losses[-1] < best_val:
        best_val = val_losses[-1]
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch + 1:2d}: train loss {train_losses[-1]:.5f} | val loss {val_losses[-1]:.5f}")

model.load_state_dict(best_state)       # keep the best epoch
torch.save(model.state_dict(), "stock_lstm.pt")

plt.plot(train_losses, label="train")
plt.plot(val_losses, label="validation")
plt.title("MSE loss per epoch")
plt.xlabel("Epoch")
plt.legend()
plt.show()

## 6. Evaluate the Model

In [ ]:
model.eval()
preds = []
with torch.no_grad():
    for xb, _ in test_loader:
        preds.append(model(xb.to(device)).cpu().numpy())
pred_scaled = np.concatenate(preds)

y_true = df["Target"].iloc[val_end:].to_numpy()
y_pred = target_scaler.inverse_transform(pred_scaled.reshape(-1, 1)).ravel()

print(f"Test R²:  {r2_score(y_true, y_pred):.4f}")
print(f"Test MAE: {mean_absolute_error(y_true, y_pred):.2f} USD")

# Naive baseline: tomorrow's close = today's close
naive = df["Close"].iloc[val_end:].to_numpy()
print(f"Naive baseline – R²: {r2_score(y_true, naive):.4f} | MAE: {mean_absolute_error(y_true, naive):.2f} USD")

plt.figure(figsize=(13, 4))
plt.plot(df.index[val_end:], y_true, label="actual next-day close")
plt.plot(df.index[val_end:], y_pred, label="LSTM prediction")
plt.title(f"{TICKER}: test period")
plt.legend()
plt.show()

In [ ]:
# Save the scaler objects for future predictions
with open("stock_scalers.pkl", "wb") as f:
    pickle.dump({"feature_scaler": feature_scaler, "target_scaler": target_scaler}, f)
print("Saved stock_lstm.pt and stock_scalers.pkl")

## Interpretation
- **R²** measures how much of the variation in next-day prices the model explains. Stock prices change slowly from day to day, so R² is usually **high (often > 0.9)**. That sounds impressive, but most of it comes from simply knowing today's price.
- That's why the notebook compares against the **naive baseline** ("tomorrow = today"), which is surprisingly hard to beat. If the LSTM's MAE isn't clearly lower than the baseline's, it hasn't learned anything beyond "prices are close to yesterday's".
- A common issue: if the test period's prices are **higher than anything in the training period** (Apple rose a lot), MinMax-scaled values go above 1 and the model tends to **under-predict**. Predicting daily *returns* (percentage change) instead of raw prices is a common fix.
- Takeaway: LSTMs can model sequences, but stock prices are close to a random walk, so this is a great exercise in **honest evaluation**, not a trading strategy.